# 🧠 Long-Term Semantic Memory

Semantic memory stores **timeless facts** about a user ("Alice is a data engineer," "prefers
Python over Scala") that should be available across *every* future thread. In LangGraph this is
the canonical use case for the cross-thread **`Store`** — no checkpointer involved, since
checkpointers are thread-scoped by design.

This notebook builds a real tool-calling LangGraph agent: the LLM itself decides when to save a
fact and when to recall one, via tools backed by `SqliteStore`, namespaced by `user_id`. The tools
reach the store — and the trusted `user_id` — through LangChain's `ToolRuntime`.

## Learning Objectives
In this notebook, you will learn:
1. **Memory tools backed by the store** - tools read the store and the trusted `user_id` from `ToolRuntime`
2. **Preload + tools** - put known facts in the system prompt, keep tools for new ones
3. **Cross-thread recall and user isolation** - facts follow the user, not the thread
4. **Update in place** - a stable key per fact turns a changed fact into an overwrite, not a duplicate

## Prerequisites
- `02_session_memory_sqlite.ipynb`, and `01_cross_thread_memory_store.ipynb` for the `Runtime` + `context` pattern
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

---
## 📦 Step 1: Environment Setup

The setup cell loads the API keys from the project-root `.env` and silences library deprecation warnings.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: decides when to save a fact and answers from the facts it knows
# ============================================================================
from aied_helpers import get_openai_llm

llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ LLM initialized: {llm.model_name}")

---
## 🛠️ Part 1: Memory Tools Backed by the Store

A tool that declares a `runtime: ToolRuntime[Context]` parameter gets it injected when `ToolNode` runs
the tool. `runtime.store` is the store the graph was compiled with, and `runtime.context` is the context
passed to `invoke()`. The parameter is not part of the tool's schema, so the model never sees it and has
no way to fill it in.

That is how the `user_id` stays out of the model's hands. It comes from `context` (trusted, server-side),
and the tools take no `user_id` argument at all — so the model can't make up a different one in a tool
call and read someone else's memory.

Each fact is saved under a short, fixed `key`, like `"name"` or `"preferred_language"`. If the
same fact gets saved again later, it overwrites that same entry instead of adding a new one.
Skip this and facts start piling up as duplicates — Part 4 checks it.

In [ ]:
# ============================================================================
# PART 1: A SEMANTIC STORE AND TWO MEMORY TOOLS
# ============================================================================
import json
import os
import sqlite3
from dataclasses import dataclass

from langchain.tools import ToolRuntime, tool
from langgraph.store.sqlite import SqliteStore

DB_PATH = "semantic_memory.db"
RESET_DB = True  # False keeps memories from earlier runs
if RESET_DB and os.path.exists(DB_PATH):
    os.remove(DB_PATH)
conn = sqlite3.connect(DB_PATH, check_same_thread=False, isolation_level=None)
semantic_store = SqliteStore(conn)  # no TTL -- semantic memory is meant to be durable
semantic_store.setup()


@dataclass
class Context:
    user_id: str  # trusted: set by your server from the login, never by the model


@tool
def save_user_fact(key: str, fact: str, runtime: ToolRuntime[Context]) -> str:
    """Persist a durable fact about the user under a short stable key
    (e.g. key='name', key='preferred_language'). Reuse a key to update that fact."""
    runtime.store.put(("semantic", runtime.context.user_id), key, {"fact": fact})
    return f"Saved fact [{key}]: {fact}"


@tool
def recall_user_facts(runtime: ToolRuntime[Context]) -> str:
    """Return every stored fact about the current user."""
    items = runtime.store.search(("semantic", runtime.context.user_id))
    facts = {item.key: item.value["fact"] for item in items}
    return json.dumps(facts) if facts else "No facts stored yet."


semantic_tools = [save_user_fact, recall_user_facts]

# The model sees only `key` and `fact`: runtime is injected, not part of the tool's schema
print("save_user_fact arguments the model sees:", list(save_user_fact.tool_call_schema.model_json_schema()["properties"]))
print("✅ Semantic store and tools ready")

---
## 📥 Part 2: The Agent — Preload + Tool Access

Rather than making the model call `recall_user_facts` on every turn, the agent node preloads
everything already known about the user straight from the store into the system prompt. Tools
stay available for the model to use when it learns something *new*. The preload shows each fact
with its key in brackets, so the model can reuse the key when a fact changes.

The graph has no checkpointer on purpose: every call is a brand-new conversation, so anything the
agent remembers between calls can only come from the store.

In [ ]:
# ============================================================================
# PART 2: THE MEMORY-AWARE AGENT GRAPH
# ============================================================================
from langchain_core.messages import HumanMessage, SystemMessage
from langgraph.graph import END, START, MessagesState, StateGraph
from langgraph.prebuilt import ToolNode, tools_condition
from langgraph.runtime import Runtime

llm_with_tools = llm.bind_tools(semantic_tools)

SYSTEM_TEMPLATE = """You are a helpful assistant with persistent semantic memory.

Known facts about this user ([key] fact):
{facts}

When the user reveals a durable fact about themselves (name, role, preference), call
`save_user_fact` with a short stable key. When a known fact changes, reuse its [key].
"""


def agent_node(state: MessagesState, runtime: Runtime[Context]) -> dict:
    items = runtime.store.search(("semantic", runtime.context.user_id))
    facts_block = "\n".join(f"- [{it.key}] {it.value['fact']}" for it in items) or "(none)"

    system = SystemMessage(SYSTEM_TEMPLATE.format(facts=facts_block))
    response = llm_with_tools.invoke([system] + state["messages"])
    return {"messages": [response]}


builder = StateGraph(MessagesState, context_schema=Context)
builder.add_node("agent", agent_node)
builder.add_node("tools", ToolNode(semantic_tools))
builder.add_edge(START, "agent")
builder.add_conditional_edges("agent", tools_condition, ["tools", END])
builder.add_edge("tools", "agent")

semantic_agent = builder.compile(store=semantic_store)


def chat(text: str, user_id: str) -> None:
    """One brand-new conversation for `user_id`: prints the tool calls and the answer."""
    result = semantic_agent.invoke({"messages": [HumanMessage(text)]}, context=Context(user_id=user_id))
    calls = [f"{tc['name']}({tc['args']})" for m in result["messages"] for tc in getattr(m, "tool_calls", [])]
    print(f"[{user_id}] USER: {text}")
    print("  tool calls:", calls or "none")
    print("  AGENT:", result["messages"][-1].content, "\n")


print("✅ Semantic-memory agent compiled")

---
## 🧵 Part 3: Cross-Thread Recall and User Isolation

### 3.1 Conversation 1 — The Agent Learns About Alice

In [ ]:
# ============================================================================
# PART 3.1: TEACH THE AGENT
# ============================================================================
USER_ID = "user-alice"
chat("Hi, I'm Alice. I work as a data engineer and prefer Python over Scala.", USER_ID)

### 3.2 Conversation 2 — Brand-New Conversation, Same User

Short-term memory (the checkpointer) isn't even wired up in this graph — there's nothing to
carry over between conversations except what the **store** persists. This is the cleanest possible
proof that semantic memory survives independently of any thread.

In [ ]:
# ============================================================================
# PART 3.2: NEW CONVERSATION, MEMORY PERSISTS
# ============================================================================
chat("What's my name and which language do I prefer?", USER_ID)

### 3.3 User Isolation

A different `user_id` — even mid-way through the exact same process — sees nothing.

In [ ]:
# ============================================================================
# PART 3.3: ISOLATION CHECK
# ============================================================================
chat("What do you know about me?", "user-bob")

---
## ✏️ Part 4: Update in Place

Alice changes her language preference. If the model reuses the fact's key, `store.put` overwrites
that item instead of adding a second, contradictory one. The check reads the namespace with
`store.search()` — the public API — before and after the conversation, and compares in code.

In [ ]:
# ============================================================================
# PART 4: A CHANGED FACT OVERWRITES ITS KEY
# ============================================================================
def facts_of(user_id: str) -> dict:
    return {it.key: it.value["fact"] for it in semantic_store.search(("semantic", user_id))}


before = facts_of(USER_ID)
chat("Quick update: I've switched, and I now prefer Rust over Python.", USER_ID)
after = facts_of(USER_ID)

print(f"facts before: {len(before)}, after: {len(after)}")
for key, fact in after.items():
    note = f"   (was: {before[key]})" if key in before and before[key] != fact else ""
    print(f"  [{key}] {fact}{note}")

updated = [key for key in after if key in before and before[key] != after[key]]
if updated and len(after) == len(before):
    print(f"\n✅ [{updated[0]}] was updated in place -- no duplicate, no contradiction")
else:
    print("\n⚠️ The model wrote a new key instead of reusing one: two facts now disagree (see Gotchas)")

---
## 📝 Summary

In this notebook, we learned:

### 1. Memory tools backed by the store
- `runtime: ToolRuntime[Context]` gives a tool the graph's store (`runtime.store`) and the trusted context
  (`runtime.context.user_id`); it is hidden from the model's tool schema
- Semantic memory = a `Store` (here, `SqliteStore`, no TTL) namespaced by `("semantic", user_id)`

### 2. Preload + tools
- The node preloads known facts into the system prompt; tools handle new ones

### 3. Cross-thread recall and user isolation
- A brand-new conversation recalls the facts; another `user_id` sees an empty namespace

### 4. Update in place
- A stable `key` per fact is what makes "update in place" work; showing the keys in the preload lets the
  model reuse them. The Store API doesn't enforce it for you

### 5. Gotchas
- **Namespace tuples are a design decision, not an implementation detail.** `("semantic", user_id)`
  isolates users; a global namespace, or `thread_id` instead of `user_id`, makes facts leak across users or
  vanish per thread
- **`ToolRuntime` exists only inside a run.** `ToolNode` (or an agent) injects it when it runs the tool; a
  direct `save_user_fact.invoke(...)` outside a graph has no runtime. Code outside the graph — setup, admin,
  tests — uses the store object itself, as Part 4's check does
- **Update-in-place depends entirely on a stable `key`.** If the model (or the tool) generates a fresh key per
  save instead of reusing `"name"`/`"preferred_language"`, you silently get duplicates and contradictions —
  nothing in the Store API prevents it
- **Preloading everything doesn't scale, and `search()` returns 10 items by default.** A blind preload silently
  drops facts past the tenth (pass `limit=`). At hundreds of facts, use `query=` on a store with an embedding
  `index` instead
- Trusted ids and SQLite concurrency apply to every layer: see [`memory_layers_guide.md`](../assets/memory_layers_guide.md)

### Next Steps
- **`04_long_term_episodic_memory_sqlite.ipynb`**: time-stamped experiences instead of timeless facts